# Workshop 2: DeepConvLSTM - guided Google Colab notebook

Start with the reference, then run one experiment part at a time.

**Before running:**
1. Upload the updated `deepconvlstm.zip` to the top level of Google Drive (`MyDrive`).
2. Select a GPU runtime (T4 if available).
3. Run the cells below in order. Data and results are saved under `MyDrive/deepconvlstm_workshop`.

There are **13 runs**. Part D needs one additional comparison baseline because the supplied reference already has weight decay and augmentation disabled. See the explanation in Part D below.

## 1. Connect Drive, unpack the code, and check the GPU
Colab will ask you to connect your own Google account. The code is unpacked into a separate workshop folder. Existing processed data and runs in that folder are kept. Re-running this setup refreshes the code from the uploaded ZIP.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path, PurePosixPath
from zipfile import ZipFile
import os, subprocess, sys

ARCHIVE = Path('/content/drive/MyDrive/deepconvlstm.zip')
PROJECT = Path('/content/drive/MyDrive/deepconvlstm_workshop')
if not ARCHIVE.is_file():
    raise FileNotFoundError('Upload the updated deepconvlstm.zip to the top level of MyDrive first.')
PROJECT.mkdir(parents=True, exist_ok=True)
with ZipFile(ARCHIVE) as archive:
    for entry in archive.infolist():
        if entry.is_dir():
            continue
        relative = PurePosixPath(entry.filename).relative_to('deepconvlstm')
        if relative.is_absolute() or '..' in relative.parts:
            raise ValueError('Invalid path in ZIP')
        destination = PROJECT.joinpath(*relative.parts)
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(archive.read(entry))
os.chdir(PROJECT)
for required in ('workshop.py', 'losses.py'):
    if not Path(required).is_file():
        raise FileNotFoundError('This looks like the original ZIP. Upload the updated package provided with this notebook.')
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
import torch
print('Project:', PROJECT)
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'not enabled')
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime, reconnect, then run this setup cell again.')

## 2. Download and prepare OPPORTUNITY
Only the first session downloads and processes the data. This keeps the supplied training/validation/test split, fills missing readings, and fits normalization using training data only.

In [ ]:
if not Path('data/opportunity.npz').is_file():
    subprocess.run([sys.executable, 'download_data.py', '--out', '/content/opportunity_raw'], check=True)
    subprocess.run([sys.executable, 'preprocess.py', '--raw', '/content/opportunity_raw',
                    '--out', 'data/opportunity.npz'], check=True)
else:
    print('Reusing data/opportunity.npz from Drive.')
print('Processed data size:', round(Path('data/opportunity.npz').stat().st_size / 1024**2, 1), 'MiB')

## 3. Check the new options and choose the training budget
The fast checks verify implementation correctness; they are not activity-recognition results.

Use the same settings for every final comparison. The reference uses 100 maximum epochs and stops early after 20 epochs without validation weighted-F1 improvement. Keep `EPOCHS = 100` for the final runs; shorter debugging runs should use separate names/seeds and must not be compared with full runs.

In [ ]:
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], check=True)
subprocess.run([sys.executable, 'workshop.py', '--list'], check=True)

EPOCHS = 100
SEED = 0

def run_part(part):
    subprocess.run([sys.executable, 'workshop.py', '--part', part,
                    '--epochs', str(EPOCHS), '--seed', str(SEED)], check=True)

## 4. Train the reference (Part 0)
This uses all 113 channels, 24-sample windows (0.8 seconds), four convolutional layers, two LSTM layers, dropout 0.5, RMSProp, and ordinary cross entropy. It includes Null.

An epoch is one pass through the training windows. A star in the log marks a new best validation score. The best checkpoint is selected using validation data; test scores are calculated afterwards.

In [ ]:
run_part('ref')

### Read the reference results
Weighted F1 gives more influence to frequent classes. Macro F1 gives every class equal weight. Inspect both, then inspect per-gesture F1 in `classification_report.txt`.

In [ ]:
import json
from IPython.display import display, Image
reference = Path('runs') / f'ref_s{SEED}'
metrics = json.loads((reference / 'metrics.json').read_text())
for key in ('params', 'best_epoch', 'train_minutes', 'test_f1_weighted', 'test_f1_macro', 'test_accuracy'):
    print(key, ':', metrics[key])
print((reference / 'classification_report.txt').read_text())
display(Image(filename=str(reference / 'learning_curves.png')))
display(Image(filename=str(reference / 'confusion_matrix.png')))

## 5. Part A - does recurrence help?
Two runs: a CNN with ordinary dense layers, and a CNN with GRU instead of LSTM. Compare both with `ref`. The supplied model shapes and training settings are retained.

In [ ]:
run_part('A')

## 6. Part B - do convolutional layers help?
Two runs: one convolutional layer and two convolutional layers, compared with the four-layer reference. All other settings remain the same.

In [ ]:
run_part('B')

## 7. Part C - how much past movement is needed?
Two runs: 20 samples (0.67 seconds) and 60 samples (2 seconds). The window step remains at 12 samples, so only window length is changed. Compare with the reference's 24 samples.

In [ ]:
run_part('C')

## 8. Part D - remove one regularizer at a time
**Why an extra baseline is needed:** the supplied reference already has weight decay=0 and augmentation off. Removing them from it would repeat the reference.

This cell first trains `D_ref`: dropout=0.5, weight decay=0.0001, augmentation on. It then trains three variants, each removing exactly one of those settings. **Compare all three with `D_ref`, not with `ref`.** State this extra baseline and the chosen weight decay in your report.

Inspect the loss curves and test scores. Removing a regularizer is not guaranteed to hurt performance.

In [ ]:
run_part('D')

## 9. Part E - learning-rate warm-up
The learning rate used in epochs 1-5 is 0.0002, 0.0004, 0.0006, 0.0008, and 0.001. It stays at 0.001 afterwards. All other settings match the reference. Compare loss fluctuations and final scores; use `history.csv` to inspect the learning rates.

In [ ]:
run_part('E')

## 10. Part F - Focal Loss
This changes only the loss to multiclass Focal Loss with gamma=2. It downweights easy examples, so relatively difficult examples contribute more strongly. It does not guarantee improved rare-gesture recognition.

Compare macro F1 and individual rare gestures with `ref`, alongside weighted F1. Checkpoint selection stays on validation weighted F1 to keep this a one-factor experiment. The plotted loss curves use cross entropy for both training and validation; the separate `train_loss` column records the focal objective.

In [ ]:
run_part('F')

## 11. Save and inspect the comparison table
This table contains only completed workshop runs, with the correct baseline for every comparison. Empty deltas mean the comparison baseline is not finished or the row itself is a baseline. F1 is on a 0-1 scale; a difference of 0.01 is one percentage point.

In [ ]:
subprocess.run([sys.executable, 'summarize.py'], check=True)
subprocess.run([sys.executable, 'workshop.py', '--summary'], check=True)
import pandas as pd
if Path('workshop_comparison.csv').is_file():
    results = pd.read_csv('workshop_comparison.csv')
    display(results.round(4))
    print('Saved on Drive:', PROJECT / 'workshop_comparison.csv')

## 12. Reload a checkpoint and save its predictions
This evaluates the already selected reference checkpoint. It does not train a new model.

In [ ]:
subprocess.run([sys.executable, 'evaluate.py', '--run', f'runs/ref_s{SEED}',
                '--save-predictions'], check=True)

## After a Colab disconnection
Reconnect the GPU runtime, then run setup, the data check, and the settings cell. Re-run the desired part. Completed runs with matching settings are skipped. A run interrupted before completion restarts from epoch 1; saved checkpoints do not include optimizer state for resuming.

Your code/results folder is `MyDrive/deepconvlstm_workshop`. Keep code and the joint report on GitHub, as requested by the workshop. The included `.gitignore` excludes raw data and checkpoint weights.

The original A-H `ablations.sh` is a larger general-purpose suite whose letters do not match this workshop. Use this notebook and `workshop.py` for the requested A-F plan.